# Interactive Tutorial: Morpho-Spectral Projection Analysis (`MSPA`) for Plant Spectroscopy

This notebook provides a step-by-step didactic tour of **Morpho-Spectral Projection Analysis (`MSPA`)** across the optical vegetation domain (**VNIR–SWIR, 400–2500 nm**).

### What You Will Learn in This Notebook:
1. **Continuous Foliar Reflectance ($\rho(\lambda) \in [0, 1]$, 400–2500 nm):** Generating Savitzky–Golay smoothed leaf-clip reflectance signatures ($D = 2,101$ continuous bands) and visualizing the 5 biochemical absorption valleys ($\mathcal{V}_1\text{--}\mathcal{V}_5$) and 8 radiative sub-windows ($\mathcal{W}_1\text{--}\mathcal{W}_8$).
2. **Breaking Column-Permutation Invariance:** Proving numerically why conventional reducers (`PCA`, `PLS-DA`) are blind to wavelength ordering while `MSPA` captures sequential curve geometry.
3. **Inspecting the 64 Semantic Variables & Module 1 Mechanics:** Exploring `.set_output(transform='pandas')`, variable cataloging, additive continuum removal ($D_{\max}, A_{\text{abs}}$), and Red-Edge inflection kinematics ($\lambda_{\text{RE}}$).
4. **Multi-Order Structure Loadings ($\mathbf{\Lambda}_{\text{MSPA}}$) & Fisher Weights ($\omega_q$):** Visualizing how `MSPA` localizes 0th-, 1st-, and 2nd-derivative features across Modules 1, 2, and 3.
5. **Autonomous Kneedle Breaking-Point (`MSPA(n_components='auto')`) & 2D Subspace Geometry:** Comparing class separability ($J_F$) across `PCA`, `PLS-DA`, and `MSPA`.
6. **5-Fold Cross-Validation Benchmark & Internal Module Ablation:** Evaluating isolated and combined `MSPA` modules across `Random Forest`, `SVM-RBF`, and `LDA`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score

try:
    from mspa import MSPA, generate_synthetic_vegetation_spectra, kneedle_breaking_point
except ImportError:
    !pip install -q --upgrade --force-reinstall git+[https://github.com/Lucas-Osco/mspa.git](https://github.com/Lucas-Osco/mspa.git)
    from mspa import MSPA, generate_synthetic_vegetation_spectra, kneedle_breaking_point

# Mentone Sunset palette used throughout the manuscript
C_PAL = ["#48385f", "#995375", "#db4a5b", "#fc9a59", "#fee086"]
CMAP_SUNSET = LinearSegmentedColormap.from_list("mentone_sunset", C_PAL, N=256)
plt.rcParams.update({"font.size": 9.5, "axes.titlesize": 10.5, "axes.titleweight": "bold"})
print("[OK] MSPA environment loaded successfully.")

---
## 1. Continuous Foliar Reflectance Spectra ($\rho(\lambda) \in [0, 1]$, 400–2500 nm) & Radiative Windows

We generate $N = 450$ continuous Savitzky–Golay smoothed leaf-clip reflectance spectra ($D = 2,101$ bands from $400\text{ to }2500\text{ nm}$, including the liquid water absorption bands at $1450\text{ nm}$ and $1940\text{ nm}$) spanning three physiological states:
- **Healthy Control:** Baseline chlorophyll absorption and mesophyll/SWIR structure.
- **Chlorotic Stress:** Pigment degradation inducing a subtle $\sim 2.8\text{ nm}$ Red-Edge blue-shift.
- **Structural / Hydric Stress:** Altered lignin/cellulose concavity in SWIR 1 ($1685\text{ nm}$) and SWIR 2 ($2180\text{ nm}$).

In [ ]:
def build_continuous_leafclip_spectra(n_samples: int = 450, random_state: int = 42):
    """Generates continuous 400-2500 nm (D = 2,101 bands) leaf-clip reflectance spectra in [0, 1]."""
    rng = np.random.default_rng(random_state)
    wls_full = np.arange(400.0, 2501.0, 1.0)
    n_full = len(wls_full)
    y_out = np.repeat([0, 1, 2], n_samples // 3)
    if len(y_out) < n_samples:
        y_out = np.concatenate([y_out, np.zeros(n_samples - len(y_out), dtype=int)])
    y_out = rng.permutation(y_out)

    X_raw = np.zeros((n_samples, n_full), dtype=np.float64)
    for i in range(n_samples):
        cls = y_out[i]
        nir_albedo = rng.normal(0.50, 0.04)
        global_tilt = rng.normal(0.0, 0.008) * ((wls_full - 1200.0) / 1000.0)
        re_center = 716.0 - (2.8 if cls == 1 else (0.8 if cls == 2 else 0.0)) + rng.normal(0, 0.9)
        re_width = 14.5 + (2.2 if cls == 1 else (-1.2 if cls == 2 else 0.0)) + rng.normal(0, 0.5)
        chl_depth = 0.115 - (0.018 if cls == 1 else 0.0) + rng.normal(0, 0.006)
        swir1_lignin_dip = 0.042 + (0.020 if cls == 2 else 0.0) + rng.normal(0, 0.004)
        swir1_dip_width = 22.0 if cls != 2 else 14.0
        swir2_cell_dip = 0.028 + (0.012 if cls == 1 else (-0.010 if cls == 2 else 0.0)) + rng.normal(0, 0.003)

        green_peak = chl_depth * np.exp(-0.5 * ((wls_full - 552.0) / 24.0) ** 2)
        chl_well = -0.035 * np.exp(-0.5 * ((wls_full - 672.0) / 18.0) ** 2)
        red_edge_ramp = nir_albedo / (1.0 + np.exp(-(wls_full - re_center) / re_width))
        swir_decay = 1.0 - 0.36 / (1.0 + np.exp(-(wls_full - 1380.0) / 85.0)) - 0.38 / (1.0 + np.exp(-(wls_full - 1880.0) / 75.0))

        w1_abs = 0.032 * np.exp(-0.5 * ((wls_full - 975.0) / 28.0) ** 2)
        w2_abs = 0.052 * np.exp(-0.5 * ((wls_full - 1195.0) / 34.0) ** 2)
        h2o_1450 = 0.165 * np.exp(-0.5 * ((wls_full - 1450.0) / 42.0) ** 2)
        lignin_1685 = swir1_lignin_dip * np.exp(-0.5 * ((wls_full - 1685.0) / swir1_dip_width) ** 2)
        h2o_1940 = 0.145 * np.exp(-0.5 * ((wls_full - 1940.0) / 45.0) ** 2)
        cell_2180 = swir2_cell_dip * np.exp(-0.5 * ((wls_full - 2180.0) / 20.0) ** 2)

        X_raw[i] = (
            0.045 + green_peak + chl_well + red_edge_ramp * swir_decay
            - w1_abs - w2_abs - h2o_1450 - lignin_1685 - h2o_1940 - cell_2180
            + global_tilt + rng.normal(0, 0.0028, size=n_full)
        )

    X_sg = savgol_filter(X_raw, window_length=11, polyorder=2, deriv=0, axis=1)
    X_refl = np.clip(X_sg, 0.005, 0.995)
    cols = [f"WL_{int(w)}" for w in wls_full]
    return pd.DataFrame(X_refl, columns=cols), y_out, wls_full

X_df, y, wavelengths = build_continuous_leafclip_spectra(n_samples=450, random_state=42)
class_names = {
    0: "Healthy Control",
    1: "Chlorotic Stress (Red-Edge Blue-Shift)",
    2: "Structural Stress (SWIR Concavity Shift)",
}
cls_colors = [C_PAL[0], C_PAL[2], C_PAL[3]]

fig, ax = plt.subplots(figsize=(13.0, 4.5), dpi=140)
for c_id, c_label in class_names.items():
    X_c = X_df.to_numpy()[y == c_id]
    mu_c, sd_c = X_c.mean(axis=0), X_c.std(axis=0)
    ax.plot(wavelengths, mu_c, color=cls_colors[c_id], lw=1.9, label=c_label)
    ax.fill_between(wavelengths, mu_c - sd_c, mu_c + sd_c, color=cls_colors[c_id], alpha=0.14)

# Highlight the 5 major biochemical absorption valleys (V1 to V5) of Module 1
valleys_viz = [
    (560, 740, "V1: Chl"), (910, 1060, "V2: NIR-W1"), (1130, 1280, "V3: NIR-W2"),
    (1520, 1760, "V4: SWIR 1"), (2050, 2360, "V5: SWIR 2")
]
for v_start, v_end, v_lbl in valleys_viz:
    ax.axvspan(v_start, v_end, color=C_PAL[4], alpha=0.22)
    ax.text(0.5 * (v_start + v_end), 0.61, v_lbl, ha="center", va="center", fontsize=8.2, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="#BBBBBB"))

ax.set_xlim(400, 2500); ax.set_ylim(0.0, 0.66)
ax.set_title(f"Continuous Leaf-Clip Reflectance Spectra ρ(λ) ∈ [0, 1] (400–2500 nm, D = {len(wavelengths)} Bands)")
ax.set_xlabel("Wavelength λ (nm)"); ax.set_ylabel("Reflectance Factor ρ(λ) [0–1]")
ax.legend(loc="upper right", frameon=True); ax.grid(True, ls="--", alpha=0.35)
plt.tight_layout(); plt.show()

---
## 2. Testing Column-Permutation Invariance (`PCA` vs. `MSPA`)

If we randomly shuffle the $2,101$ wavelength columns ($\mathbf{X}_{\mathbf{\Pi}} = \mathbf{X}\mathbf{\Pi}$), `PCA` produces the **exact same subspace scores** ($\approx 10^{-15}$ numerical precision), because global covariance matrices ignore physical band adjacency. In contrast, `MSPA` relies on local Savitzky–Golay derivatives, continuum chords, and contiguous sub-windows, immediately detecting the broken curve topology.

In [ ]:
rng = np.random.default_rng(777)
perm_indices = rng.permutation(X_df.shape[1])
X_shuffled = X_df.to_numpy()[:, perm_indices]

Z_pca_orig = PCA(n_components=10, random_state=42).fit_transform(X_df.to_numpy())
Z_pca_shuf = PCA(n_components=10, random_state=42).fit_transform(X_shuffled)

Z_mspa_orig = MSPA(n_components=64, wavelengths=wavelengths).fit_transform(X_df.to_numpy(), y)
Z_mspa_shuf = MSPA(n_components=64, wavelengths=wavelengths).fit_transform(X_shuffled, y)

print(f"Max |Z_PCA(X)  - Z_PCA(X*Pi)|  : {np.max(np.abs(Z_pca_orig - Z_pca_shuf)):.2e}  --> Column-permutation INVARIANT (Order-blind)")
print(f"Mean |Z_MSPA(X) - Z_MSPA(X*Pi)|: {np.mean(np.abs(Z_mspa_orig - Z_mspa_shuf)):.4f}  --> Physically sensitive to wavelength contiguity")

---
## 3. Fitting `MSPA`, Inspecting the 64 Semantic Variables & Module 1 Mechanics

Using `.set_output(transform='pandas')`, we can inspect all $64$ extracted variables across:
- **Module 1 (`MSPA_BIO_*`, 16 variables):** 10 continuum-removed valley depths/areas ($\mathcal{V}_1\text{--}\mathcal{V}_5$), 4 red-edge inflection kinematics ($\lambda_{\text{RE}}, S_{\max}, \bar{S}, C_{\text{amp}}$), and 2 plateau contrasts.
- **Module 2 (`MSPA_WIN_*`, 24 variables):** 24 local SVD modes (Orders 0, 1, 2 across windows $\mathcal{W}_1\text{--}\mathcal{W}_8$).
- **Module 3 (`MSPA_CAN_*`, 24 variables):** 24 global Fisher-weighted canonical variables.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.25, stratify=y, random_state=42)

mspa = MSPA(n_components=64).set_output(transform="pandas")
Z_train_df = mspa.fit_transform(X_train, y_train)
Z_test_df  = mspa.transform(X_test)

# Reorder columns didactically: Module 1 (BIO) -> Module 2 (WIN) -> Module 3 (CAN)
mod1_cols = [c for c in Z_train_df.columns if c.startswith("MSPA_BIO_")]
mod2_cols = [c for c in Z_train_df.columns if c.startswith("MSPA_WIN_")]
mod3_cols = [c for c in Z_train_df.columns if c.startswith("MSPA_CAN_")]
ordered_cols = mod1_cols + mod2_cols + mod3_cols
Z_train_df = Z_train_df[ordered_cols]
Z_test_df  = Z_test_df[ordered_cols]

# Build variable catalog with Training Fisher Scores
Z_tr_np = Z_train_df.to_numpy()
f_scores = MSPA._compute_fisher_scores(Z_tr_np, y_train)
catalog_rows = []
for idx, (col_nm, f_sc) in enumerate(zip(ordered_cols, f_scores)):
    mod_lbl = "Module 1 (Biophysical Hull)" if "BIO_" in col_nm else ("Module 2 (Local Window SVD)" if "WIN_" in col_nm else "Module 3 (Global Canonical)")
    catalog_rows.append({"Index": idx + 1, "Variable": col_nm, "Module": mod_lbl, "Training_Fisher_Score": round(float(f_sc), 4)})
catalog_df = pd.DataFrame(catalog_rows)

print(f"Projected Training DataFrame Shape: {Z_train_df.shape}")
print("\n--- Top 12 Most Discriminative MSPA Variables (Ranked by Training Fisher Score) ---")
display(catalog_df.sort_values("Training_Fisher_Score", ascending=False).head(12))

# Visualize Module 1 Standardized Profile across the 3 Physiological States
short_mod1 = [c.replace("MSPA_BIO_", "").replace("Inflect_", "RE_").replace("Contrast_", "") for c in mod1_cols]
fig, ax = plt.subplots(figsize=(13.0, 4.2), dpi=140)
x_idx = np.arange(len(mod1_cols))
bw = 0.26
for c_id, c_label in class_names.items():
    mu_bio = Z_test_df.loc[y_test == c_id, mod1_cols].mean(axis=0).to_numpy()
    ax.bar(x_idx + (c_id - 1) * bw, mu_bio, width=bw, color=cls_colors[c_id], edgecolor="#222222", label=c_label)

ax.axvline(9.5, color="#555555", ls="--", lw=1.2)
ax.axvline(13.5, color="#555555", ls="--", lw=1.2)
ax.set_xticks(x_idx); ax.set_xticklabels(short_mod1, rotation=30, ha="right", fontweight="bold")
ax.set_title("Module 1 Output Profile (Variables 1–16: Continuum Valleys, Red-Edge Kinematics & Plateau Contrasts)")
ax.set_ylabel("Standardized Value (Z-Score)"); ax.legend(loc="upper left"); ax.grid(True, axis="y", ls="--", alpha=0.35)
plt.tight_layout(); plt.show()

---
## 4. Multi-Order Structure Loadings ($\mathbf{\Lambda}_{\text{MSPA}}$) & Fisher Channel Weights ($\omega_q$)

Below we compute the correlation loading matrices $\mathbf{\Lambda}^{(0)}$ (amplitude), $\mathbf{\Lambda}^{(1)}$ (1st derivative slope), $\mathbf{\Lambda}^{(2)}$ (2nd derivative concavity), and the composite footprint $|\mathbf{\Lambda}^{\text{comp}}| \in \mathbb{R}^{D \times 64}$ across the continuous $400\text{--}2500\text{ nm}$ spectrum.

In [ ]:
X_tr_arr = X_train.to_numpy(dtype=np.float64)
X_d1, X_d2 = mspa._compute_piecewise_derivatives(X_tr_arr)
Z_std = StandardScaler().fit_transform(Z_train_df.to_numpy(dtype=np.float64))
n_s = max(1, X_tr_arr.shape[0] - 1)

L0 = (StandardScaler().fit_transform(X_tr_arr).T @ Z_std) / n_s
L1 = (StandardScaler().fit_transform(X_d1).T @ Z_std) / n_s
L2 = (StandardScaler().fit_transform(X_d2).T @ Z_std) / n_s
L_comp = np.sqrt((L0 ** 2 + L1 ** 2 + L2 ** 2) / 3.0)

D = len(wavelengths)
w_ord0 = mspa.fisher_weights_[:D]
w_ord1 = mspa.fisher_weights_[D:2*D]
w_ord2 = mspa.fisher_weights_[2*D:3*D]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.8, 4.8), dpi=140, gridspec_kw={"width_ratios": [1.05, 1.15]})

# Panel A: Continuous Multi-Order Fisher Weights omega_q across 400-2500 nm
ax1.plot(wavelengths, w_ord0, color=C_PAL[0], lw=1.4, label="Order 0 Weight ω^(0) (Amplitude)")
ax1.plot(wavelengths, w_ord1, color=C_PAL[1], lw=1.5, ls="--", label="Order 1 Weight ω^(1) (1st Deriv. Slope)")
ax1.plot(wavelengths, w_ord2, color=C_PAL[2], lw=1.4, ls="-.", label="Order 2 Weight ω^(2) (2nd Deriv. Concavity)")
ax1.axhline(1.0, color="#666666", ls=":", lw=1.0)
ax1.set_xlim(400, 2500)
ax1.set_title("A. Multi-Order Fisher Discriminability Weights (ω_q)")
ax1.set_xlabel("Wavelength λ (nm)"); ax1.set_ylabel("Normalized Fisher Weight ω_q")
ax1.legend(loc="upper right", fontsize=8.2); ax1.grid(True, ls="--", alpha=0.35)

# Panel B: Complete 64 x 2,101 Composite Structure Loading Matrix |Lambda_MSPA|
im = ax2.imshow(L_comp.T, aspect="auto", cmap=CMAP_SUNSET, vmin=0.0, vmax=0.85, extent=[wavelengths[0], wavelengths[-1], 64, 1])
ax2.axhline(16.5, color="white", ls="--", lw=1.5)
ax2.axhline(40.5, color="white", ls="--", lw=1.5)
ax2.set_yticks([8, 28, 52])
ax2.set_yticklabels(["Mod 1 (1–16)\nBiophysical", "Mod 2 (17–40)\nWindow SVD", "Mod 3 (41–64)\nCanonical"], fontweight="bold")
ax2.set_title(f"B. Complete 64 × {D} Spectral Structure Loading Matrix (|Λ_MSPA|)")
ax2.set_xlabel("Input Wavelength λ (nm)")
plt.colorbar(im, ax=ax2, label="Composite Loading Magnitude |Λ_comp|")
plt.tight_layout(); plt.show()

---
## 5. Autonomous Breaking-Point (`n_components='auto'`) & 2D Subspace Projections

When `MSPA(n_components='auto')` is fitted, it estimates the intrinsic canonical breaking-point $K_{\text{can}}^*$ from the cumulative Fisher-weighted singular energy curve via the Kneedle criterion. Below we plot the Kneedle curve alongside the out-of-sample 2D subspace projections of `PCA`, `PLS-DA`, and `MSPA`.

In [ ]:
mspa_auto = MSPA(n_components="auto").fit(X_train, y_train)

# Compute Kneedle cumulative singular energy curve on H_train
X_multi_sc = mspa_auto.scaler_multi_.transform(np.hstack([X_tr_arr, X_d1, X_d2]))
X_w_cent = (X_multi_sc * mspa_auto.fisher_weights_[None, :]) - mspa_auto.mu_weighted_
_, S_vals, _ = np.linalg.svd(X_w_cent, full_matrices=False)
ev_probe = (S_vals[:60] ** 2) / np.sum(S_vals ** 2)
cum_var = np.cumsum(ev_probe)
y_norm = (cum_var - cum_var[0]) / max(cum_var[-1] - cum_var[0], 1e-12)
x_norm = np.linspace(0.0, 1.0, len(y_norm))
diff_curve = y_norm - x_norm
kc = mspa_auto.n_can_

def fisher_2d_index(Z2d, y_vec):
    mu_g = Z2d.mean(axis=0)
    sb = sum(np.sum(y_vec == c) * np.sum((Z2d[y_vec == c].mean(axis=0) - mu_g) ** 2) for c in np.unique(y_vec))
    sw = sum(np.sum((Z2d[y_vec == c] - Z2d[y_vec == c].mean(axis=0)) ** 2) for c in np.unique(y_vec))
    return float(sb / max(sw, 1e-8))

sc_x = StandardScaler().fit(X_train)
X_tr_s, X_te_s = sc_x.transform(X_train), sc_x.transform(X_test)

Z_pca_te = PCA(n_components=2, random_state=42).fit(X_tr_s).transform(X_te_s)
Z_pls_te = PLSRegression(n_components=2, scale=False).fit(X_tr_s, pd.get_dummies(y_train).values).transform(X_te_s)
Z_mspa_te = Z_test_df[["MSPA_CAN_01", "MSPA_CAN_02"]].to_numpy()

fig, axes = plt.subplots(1, 4, figsize=(16.5, 4.0), dpi=140)

# Panel A: Kneedle Curve
k_idx = np.arange(1, len(y_norm) + 1)
axes[0].plot(k_idx, y_norm, color=C_PAL[0], lw=2.0, label="Cumulative Energy y_k")
axes[0].plot(k_idx, diff_curve, color=C_PAL[2], lw=1.6, ls="--", label="Kneedle Distance D(k)")
axes[0].scatter([kc], [y_norm[kc - 1]], color="#00E676", edgecolors="#111111", s=90, marker="*", zorder=5,
                label=f"Breaking-Point K_can*={kc}\n(Total K*={mspa_auto.n_components_})")
axes[0].set_title("A. Autonomous Kneedle Breaking-Point")
axes[0].set_xlabel("Canonical Eigenmode Index k"); axes[0].set_ylabel("Normalized Energy / Deflection")
axes[0].legend(loc="center right", fontsize=7.8); axes[0].grid(True, ls="--", alpha=0.35)

# Panels B, C, D: 2D Subspaces on Holdout Test Set
subspaces = [
    ("B. Unsupervised PCA (PC1 vs. PC2)", Z_pca_te),
    ("C. Supervised PLS-DA (LV1 vs. LV2)", Z_pls_te),
    ("D. Proposed MSPA (CAN 1 vs. CAN 2)", Z_mspa_te),
]
for ax, (title_s, Z_2d) in zip(axes[1:], subspaces):
    Z_n = (Z_2d - Z_2d.mean(axis=0)) / (Z_2d.std(axis=0) + 1e-8)
    jf = fisher_2d_index(Z_n, y_test)
    for c_id, c_label in class_names.items():
        pts = Z_n[y_test == c_id]
        ax.scatter(pts[:, 0], pts[:, 1], color=cls_colors[c_id], s=28, alpha=0.82, edgecolors="#222222", lw=0.4, label=c_label.split(" (")[0])
    ax.set_title(f"{title_s}\n(Holdout Fisher J_F = {jf:.2f})")
    ax.set_xlabel("Component 1 (Z-Score)"); ax.set_ylabel("Component 2 (Z-Score)")
    ax.grid(True, ls="--", alpha=0.35)
    if "MSPA" in title_s:
        ax.legend(loc="lower right", fontsize=7.8)

plt.tight_layout(); plt.show()

---
## 6. 5-Fold Cross-Validation Benchmark & Internal Module Ablation

Finally, we evaluate a 5-fold stratified cross-validation benchmark comparing **Full Spectrum ($2,101$ bands)**, **PCA ($K^*=50$)**, **PLS-DA ($K^*=15$)**, and **MSPA**—including an internal ablation of **Module 1 Only ($16$ vars)**, **Module 2 Only ($24$ vars)**, **Module 3 Only ($24$ vars)**, and **Full MSPA ($64$ vars)** across `Random Forest`, `SVM-RBF`, and `LDA`.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
methods = [
    "Full Spectrum (2,101)",
    "PCA (K* = 50)",
    "PLS-DA (K* = 15)",
    "MSPA Mod 1 Only (16)",
    "MSPA Mod 2 Only (24)",
    "MSPA Mod 3 Only (24)",
    "Full MSPA (K* = 64)",
]
records = {m: {"RF": [], "SVM-RBF": [], "LDA": []} for m in methods}

def eval_classifiers(Z_tr, y_tr, Z_te, y_te, rec_dict):
    sc = StandardScaler().fit(Z_tr)
    Ztr_s, Zte_s = sc.transform(Z_tr), sc.transform(Z_te)
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Ztr_s, y_tr)
    svm = SVC(C=1.0, kernel="rbf", random_state=42).fit(Ztr_s, y_tr)
    lda = LinearDiscriminantAnalysis(solver="svd").fit(Ztr_s, y_tr)
    rec_dict["RF"].append(f1_score(y_te, rf.predict(Zte_s), average="macro"))
    rec_dict["SVM-RBF"].append(f1_score(y_te, svm.predict(Zte_s), average="macro"))
    rec_dict["LDA"].append(f1_score(y_te, lda.predict(Zte_s), average="macro"))

for tr_idx, te_idx in skf.split(X_df, y):
    X_tr, X_te = X_df.iloc[tr_idx], X_df.iloc[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    sc = StandardScaler().fit(X_tr)
    X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)

    # 1. Full Spectrum (2,101 continuous bands)
    eval_classifiers(X_tr_s, y_tr, X_te_s, y_te, records["Full Spectrum (2,101)"])

    # 2. PCA (50)
    pca = PCA(n_components=50, random_state=42).fit(X_tr_s)
    eval_classifiers(pca.transform(X_tr_s), y_tr, pca.transform(X_te_s), y_te, records["PCA (K* = 50)"])

    # 3. PLS-DA (15)
    pls = PLSRegression(n_components=15, scale=False).fit(X_tr_s, pd.get_dummies(y_tr).values)
    eval_classifiers(pls.transform(X_tr_s), y_tr, pls.transform(X_te_s), y_te, records["PLS-DA (K* = 15)"])

    # 4. MSPA and Module Ablations
    mspa_cv = MSPA(n_components=64).set_output(transform="pandas").fit(X_tr, y_tr)
    Z_tr_df, Z_te_df = mspa_cv.transform(X_tr), mspa_cv.transform(X_te)
    c_bio = [c for c in Z_tr_df.columns if c.startswith("MSPA_BIO_")]
    c_win = [c for c in Z_tr_df.columns if c.startswith("MSPA_WIN_")]
    c_can = [c for c in Z_tr_df.columns if c.startswith("MSPA_CAN_")]

    eval_classifiers(Z_tr_df[c_bio].to_numpy(), y_tr, Z_te_df[c_bio].to_numpy(), y_te, records["MSPA Mod 1 Only (16)"])
    eval_classifiers(Z_tr_df[c_win].to_numpy(), y_tr, Z_te_df[c_win].to_numpy(), y_te, records["MSPA Mod 2 Only (24)"])
    eval_classifiers(Z_tr_df[c_can].to_numpy(), y_tr, Z_te_df[c_can].to_numpy(), y_te, records["MSPA Mod 3 Only (24)"])
    eval_classifiers(Z_tr_df.to_numpy(),        y_tr, Z_te_df.to_numpy(),        y_te, records["Full MSPA (K* = 64)"])

summary_rows = []
for m_name, d_scores in records.items():
    rf_m, svm_m, lda_m = np.mean(d_scores["RF"]), np.mean(d_scores["SVM-RBF"]), np.mean(d_scores["LDA"])
    summary_rows.append({
        "Representation": m_name,
        "Random_Forest_F1": rf_m,
        "SVM_RBF_F1": svm_m,
        "LDA_F1": lda_m,
        "Mean_Macro_F1": np.mean([rf_m, svm_m, lda_m]),
    })

summary_df = pd.DataFrame(summary_rows)
print("=== 5-Fold Cross-Validation Benchmark & Module Ablation Summary ===")
display(summary_df.round(4))